# Daily Challenge: Pinecone Serverless Reranking
This notebook demonstrates how to use Pinecone's reranking capabilities to improve search relevance, specifically in the context of clinical notes.

In [1]:
# 1. Install Pinecone libraries
!pip install -U pinecone==6.0.1 pinecone-notebooks

In [2]:
# 2. Authenticate with Pinecone
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

In [3]:
# 3. Instantiate the Pinecone client
from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

In [4]:
# 4. Define your query & documents
query = "Tell me about Apple's products"
documents = [
    "The Granny Smith is a tip-bearing apple cultivar.",
    "Apple announced the new iPhone 15 Pro with a titanium design.",
    "Red Delicious is one of the most famous American apple varieties.",
    "The MacBook Air with M2 chip offers incredible performance and battery life.",
    "Fuji apples are large and sweet, making them great for snacks."
]

In [5]:
# 5. Call the reranker
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

In [6]:
# 6. Inspect reranked results
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        # Accessing position (i+1), similarity score, and text
        print(f"Rank: {i+1} | Score: {m.score:.4f} | Text: {m.document.text}")

# The reranked object contains the results in the 'data' attribute
show_reranked_results(query, reranked.data)

Query: Tell me about Apple's products
Rank: 1 | Score: 0.0393 | Text: Apple announced the new iPhone 15 Pro with a titanium design.
Rank: 2 | Score: 0.0223 | Text: The MacBook Air with M2 chip offers incredible performance and battery life.
Rank: 3 | Score: 0.0049 | Text: Red Delicious is one of the most famous American apple varieties.


## Part 2: Setup a Serverless Index for Medical Notes

In [7]:
# 1. Install data & model libraries
!pip install pandas torch transformers

In [8]:
import time
import pandas as pd
from pinecone import ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# 2. Define environment settings
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')
spec = ServerlessSpec(cloud=cloud, region=region)
index_name = 'medical-notes-index'

# 3. Create or recreate the index
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384, # Matches MiniLM-L6-v2 dimension
    metric='cosine',
    spec=spec
)

{
    "name": "medical-notes-index",
    "metric": "cosine",
    "host": "medical-notes-index-2hygl38.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [10]:
# Part 3: Load the Sample Data
import requests
import tempfile
import os
import pandas as pd

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Corrected GitHub raw URL
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl"

    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

print("Data shape:", df.shape)
display(df.head())

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [11]:
# Part 4: Upsert Data into the Index
index = pc.Index(name=index_name)

# Upsert data using the dataframe
index.upsert_from_dataframe(df)

# Wait for availability
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
display(index.describe_index_stats())

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

Vector count:  100
Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

In [14]:
# Part 5: Query & Embedding Function
# Load model and tokenizer once outside the function to improve performance
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(input_question):
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
        # We take the mean of the token embeddings (dim=1)
        # Then we select the first (and only) batch to get a 1D vector
        embedding = model_output.last_hidden_state.mean(dim=1).squeeze()
    return embedding

question = "patient with heart conditions and chest pain"
query_vec = get_embedding(question).tolist()

# Get results from Pinecone index
# The dimension of query_vec should now be 384
results = index.query(vector=query_vec, top_k=5, include_metadata=True)
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
# Part 6: Display & Rerank Clinical Notes
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}')
        print(f' Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)

Question: 'patient with heart conditions and chest pain'

Results:
   1. ID: P001
 Score: 0.724850893
 Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   2. ID: P016
 Score: 0.536550939
 Metadata: {'condition': 'heart murmur', 'referral': 'cardiology'}

   3. ID: P0100
 Score: 0.408476293
 Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}

   4. ID: P090
 Score: 0.395597458
 Metadata: {'advice': 'stress management', 'symptoms': 'stress, burnout'}

   5. ID: P042
 Score: 0.395597458
 Metadata: {'advice': 'stress management', 'symptoms': 'stress, burnout'}



In [16]:
# Prepare documents for reranking
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

refined_query = "Is there evidence of hypertension or cardiovascular disease in the patient records?"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

def show_reranked_results_clinical(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}')
        print(f' Reranking Field: {match.document.reranking_field}')
        print('')

show_reranked_results_clinical(refined_query, reranked_results.data)

Question: 'Is there evidence of hypertension or cardiovascular disease in the patient records?'

Reranked Results:
   1. ID: P001
 Score: 0.0064631375
 Reranking Field: symptoms: chest pain; tests: EKG, stress test

   2. ID: P016
 Score: 0.00049553596
 Reranking Field: condition: heart murmur; referral: cardiology

   3. ID: P090
 Score: 4.908662e-05
 Reranking Field: advice: stress management; symptoms: stress, burnout

